# Project Overview & Objectives

Welcome to this step-by-step guide on aligning Large Language Models (LLMs) to human preferences.  

As a junior data scientist, you will often find that a model trained purely on next-token prediction (a **base model**) or even one fine-tuned on instructions (**SFT**) might not produce the best or safest responses.

---

**Objective**

Our objective in this project is to take a base model through a complete alignment pipeline:

Base Model $\rightarrow$ Non-Instruction Model $\rightarrow$ Instruction Model $\rightarrow$ Preference Model

Specifically, we will use **Direct Preference Optimization (DPO)** to teach our model to prefer high-quality pharmaceutical explanations over lower-quality ones.

---

**Model Architecture: Pros & Cons**

1. TinyLlama (1.1B)

    Pros:
    - Highly efficient  
    - Fast to train  
    - Fits easily into consumer GPU memory (especially with quantization/LoRA)  
    - Excellent for rapid prototyping  

    Cons:
    - Limited parameter count restricts complex reasoning compared to 7B+ models  

2. Direct Preference Optimization (DPO)

    Pros:
    - Bypasses the need for a separate Reward Model (unlike RLHF)  
    - Makes the training pipeline mathematically simpler  
    - Computationally cheaper  

    Cons:
    - Highly sensitive to dataset quality (requires strict **"chosen" vs. "rejected"** pairs)  
    - Sensitive to hyperparameters like the **beta penalty**

# Environment Setup

In [ ]:
# %pip install -U "trl==1.6.0" "bitsandbytes==0.46.1" "torchao==0.16.0" 

In [ ]:
import os
import torch
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from datasets import load_dataset
from trl import DPOTrainer, DPOConfig

# Disable Weights & Biases logging for this local prototyping session
os.environ["WANDB_DISABLED"] = "true"

In [ ]:
# Mount Google Drive and set up paths
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
DRIVE_DIR = Path('/content/drive/MyDrive/LLM-Fine-Tuning/DomainSpecific/assets')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print("Drive assets folder:", DRIVE_DIR)

In [ ]:
# Define the foundational model
BASE_MODEL_ID = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
INSTRUCTION_MODEL = f"{DRIVE_DIR}/instruction-tinyllama-adapter"

# Data Acquisition & Exploratory Data Analysis (EDA)

Context Block: DPO requires a specialized dataset structure containing a prompt, a chosen response, and a rejected response. Here, we load our proprietary pharmaceutical preference dataset.

In [ ]:
# Load the preference dataset specifying 'chosen' and 'rejected' behaviors
pref_dataset = load_dataset("csv", data_files=f"{DRIVE_DIR}/pharma_preference_data.csv")["train"]

# Preprocessing & Feature Engineering

Context Block: Language models require text to be converted into numerical tokens. We initialize the tokenizer associated with TinyLlama. A crucial feature engineering step here is setting the pad_token. Since base LLaMA architectures often lack a default padding token, we map it to the End-Of-Sequence (eos_token) to ensure our training batches are perfectly rectangular.

In [ ]:
# Initialize tokenizer from the base model
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)

# Ensure padding token is set for batching varying sequence lengths
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Model Development & Training

Context Block: This is the architectural core of our pipeline. When moving from an Instruction-tuned model to a Preference-aligned model, we must manage our LoRA (Low-Rank Adaptation) adapters carefully to avoid adapter stacking.

**The Right Way vs. The Wrong Way:**

- Correct (Our Method): Base Model + Merge the Stage 1 Instruction LoRA + Attach a New LoRA for Preference training.

- Incorrect: Base Model + LoRA + LoRA (Stacking adapters leads to instability and degraded performance).

We will load the base model in 8-bit precision, merge our previous instruction checkpoint, and configure the DPOTrainer.

## Step A: Load Base Model with 8-bit quantization for memory efficiency

In [ ]:
quantization_config = BitsAndBytesConfig(load_in_8bit=True)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto"
)

## Step B: Load Instruction LoRA and merge weights permanently

In [ ]:
# This incorporates SFT knowledge directly into the base weights
model_with_instruct = PeftModel.from_pretrained(base_model, INSTRUCTION_MODEL, device_map="auto", torch_dtype=torch.float16)
merged_model = model_with_instruct.merge_and_unload()

## Step C: Attach NEW LoRA adapter specifically for DPO Alignment

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,  # Rank of the update matrices
    lora_alpha=32,  # Maintain alpha = 2*r ratio
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",  # All attention layers
        "gate_proj", "up_proj", "down_proj"  # MLP layers (crucial for DPO)
    ],
    use_rslora=True,  # Rank-Stabilized LoRA: better scaling with higher ranks
    bias="none",
)

dpo_ready_model = get_peft_model(merged_model, lora_config)

## Step D: Configure and Execute DPOTrainer

In [ ]:
dpo_args = DPOConfig(
    output_dir="./tinyllama-dpo",
    num_train_epochs=3,  # Lesser epochs (DPO overfits quickly!)
    warmup_steps=0.1,  # Add warmup for stability (10% of training)
    lr_scheduler_type="cosine",  # Smooth decay instead of sudden drops
    
    # Optimization
    learning_rate=5e-6,  # Lower LR for stability (DPO is sensitive)
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,  # Effective batch 8
    max_grad_norm=0.3,  # Gradient clipping (prevents divergence)
    
    # Memory & Speed
    gradient_checkpointing=True,  # Trade compute for memory -> allows bigger batches
    bf16=True,  # Use bf16 if available (A100, H100, RTX 30/40 series), else fp16=True
    optim="paged_adamw_8bit",  # For QLoRA-style memory efficiency
    
    # DPO-specific improvements
    beta=0.1,
    loss_type="sigmoid",
    label_smoothing=0.1,  # Prevents overconfidence, improves generalization
    max_length=512,  # Explicitly set max sequence length (prevents OOM)
    
    # Data handling
    remove_unused_columns=False,  # Keep False if using custom formatting func
    dataloader_num_workers=1,  # Faster data loading
)

# Initialize the DPO trainer
trainer = DPOTrainer(
    model=dpo_ready_model,
    ref_model=None,  # trl will automatically create the reference model from the active model
    args=dpo_args,
    train_dataset=pref_dataset,
    processing_class=tokenizer, 
)

In [ ]:
# Execute preference alignment
trainer.train()

# Save the fine-tuned adapter and tokenizer
trainer.save_model(f"{DRIVE_DIR}/dpo-tinyllama-adapter")
tokenizer.save_pretrained(f"{DRIVE_DIR}/dpo-tinyllama-tokenizer")

# Evaluation & Results

Context Block: To prove our methodology, we must qualitatively evaluate how the model responds to a domain-specific prompt at three different stages of its lifecycle. We will ask it to explain the mechanism and alternative benefits of Metformin, observing how the response style evolves from generic, to instructive, to deeply aligned.

In [ ]:
# STEP 1: Load the raw base model (Non-Instruction)
non_instruction_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    device_map="auto",
    dtype=torch.float16 # Recommended for inference
)

# STEP 2: Load the Instruction Adapter and Merge it
model_with_instruct = PeftModel.from_pretrained(
    non_instruction_model, 
    INSTRUCTION_MODEL
)
instruction_model = model_with_instruct.merge_and_unload()

# STEP 3: Load the DPO Adapter on top of the Merged Model
dpo_adapter_path = f"{DRIVE_DIR}/dpo-tinyllama-adapter"

# Use PeftModel to attach the DPO adapter!
dpo_model = PeftModel.from_pretrained(
    instruction_model, 
    dpo_adapter_path
)

In [ ]:
# The evaluation prompt
test_prompt = "Explain how Metformin works in the human body and why some researchers believe it could have benefits beyond diabetes treatment."

def evaluate_model(model, tokenizer, question, stage_name):
    """Evaluates the model using raw text, matching the unformatted CSV training data."""
 
    prompt = f"Question: {question}\n\nAnswer:"
    
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    input_token_length = inputs['input_ids'].shape[1] 
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        repetition_penalty=1.1,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id  # Silences the warning
    )
    
    print(f"--- {stage_name} Output ---")
    print(f"Prompt:\n{prompt}\n")
    
    # Slice the tensor to only grab the newly generated answer
    generated_tokens = outputs[0][input_token_length:]
    answer = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    
    if not answer.strip():
        print("[Model generated an empty response. It may be suffering from mode collapse.]")
    else:
        print(f"Response:\n{answer.strip()}")
        
    print("\n" + "="*50 + "\n")


In [ ]:
# 1. Test Base Model
evaluate_model(non_instruction_model, tokenizer, test_prompt, "Base / Non-Instruction Model")

# 2. Test Instruction-Tuned Model
evaluate_model(instruction_model, tokenizer, test_prompt, "Instruction Fine-Tuned Model")

# 3. Test DPO Model
evaluate_model(dpo_model, tokenizer, test_prompt, "DPO Preference-Aligned Model")

# Conclusion & Future Work

In this workflow, we successfully navigated the complexities of multi-stage LLM alignment. By properly merging our SFT adapter before applying our DPO adapter, we maintained the structural integrity of the model's learned knowledge while shifting its generation distribution toward our preferred pharmaceutical safety standards.

**Next Steps:**

- Quantitative Evaluation: Implement automated metrics (e.g., ROUGE, BLEU) or an "LLM-as-a-Judge" pipeline (using GPT-4 or Claude 3) to score the DPO outputs against a held-out test set.

- Hyperparameter Tuning: Experiment with the beta parameter in DPOConfig. A higher beta forces the model to stay closer to the reference model, while a lower beta allows it to optimize more aggressively for the preference data, risking mode collapse.

- Dataset Scaling: Expand pharma_preference_data.csv to include adversarial edge cases (e.g., prompts asking for illicit drug synthesis) to test the DPO model's refusal capabilities.